# 4.2 训练策略 (Training Strategies)

训练策略决定了模型训练的效率和稳定性，是大规模预训练成功的关键工程要素。

本节涵盖：
- 学习率调度
- 批次调度
- 梯度累积
- 混合精度训练
- FP8训练

## 1. 学习率调度

**目的**：控制训练过程中学习率的变化

**基本原理**：训练初期使用小学习率避免梯度不稳定（预热），后期逐步降低学习率使模型收敛到更优解。

**主流策略**：
- **Warmup + Cosine Decay**：先线性增加学习率，再按余弦函数衰减，是LLM训练的标准选择
- **WSD（Warmup-Stable-Decay）**：三阶段策略，预热后保持稳定学习率，最后阶段衰减

**关键参数**：
- Warmup步数：通常为总步数的1-5%
- 峰值学习率：通常3e-4到1e-3
- 最小学习率：通常为峰值学习率的1/10到1/100

In [ ]:
import torch
import math

def cosine_schedule_with_warmup(step, warmup_steps, total_steps, peak_lr=1e-3, min_lr=1e-5):
    if step < warmup_steps:
        return peak_lr * step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

def wsd_schedule(step, warmup_steps, stable_steps, decay_steps, peak_lr=1e-3, min_lr=1e-5):
    if step < warmup_steps:
        return peak_lr * step / warmup_steps
    elif step < warmup_steps + stable_steps:
        return peak_lr
    else:
        progress = (step - warmup_steps - stable_steps) / decay_steps
        return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

total_steps = 10000
warmup_steps = 500
steps = list(range(total_steps))

cosine_lrs = [cosine_schedule_with_warmup(s, warmup_steps, total_steps) for s in steps]
wsd_lrs = [wsd_schedule(s, warmup_steps, 7000, 2500) for s in steps]

print('=== Learning Rate Schedules ===')
print(f'Total steps: {total_steps}, Warmup: {warmup_steps}')
print(f'\nCosine Decay checkpoints:')
for s in [0, 250, 500, 2000, 5000, 8000, 10000-1]:
    print(f'  Step {s:5d}: lr={cosine_lrs[s]:.6f}')

print(f'\nWSD Schedule checkpoints:')
for s in [0, 250, 500, 3000, 7500, 9000, 10000-1]:
    print(f'  Step {s:5d}: lr={wsd_lrs[s]:.6f}')

print(f'\nKey: Cosine decay is the standard for most LLMs (LLaMA, GPT).')
print(f'WSD keeps lr stable longer, which can be more efficient for large-scale training.')

## 2. 批次调度与梯度累积

**批次调度**：训练初期使用小批次获得更稳定的梯度估计，后期逐步增大批次以提高训练吞吐量。

**梯度累积**：在显存有限时模拟大批次训练。将大批次拆分为多个小批次，前向传播后不更新参数而是累积梯度，累积到目标步数后再进行一次参数更新。

**关键公式**：
- effective_batch_size = micro_batch_size × accumulation_steps × num_gpus
- 梯度累积在数学上等价于使用更大的批次
- 大模型训练通常使用梯度累积步数4-64

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

data = torch.randn(64, 64)
targets = torch.randn(64, 64)

optimizer.zero_grad()
loss_full = nn.MSELoss()(model(data), targets)
loss_full.backward()
full_grads = [p.grad.clone() for p in model.parameters()]
full_loss = loss_full.item()

optimizer.zero_grad()
accum_steps = 4
micro_bs = 16
accum_loss = 0.0
for i in range(accum_steps):
    start = i * micro_bs
    end = start + micro_bs
    micro_loss = nn.MSELoss()(model(data[start:end]), targets[start:end])
    micro_loss = micro_loss / accum_steps
    micro_loss.backward()
    accum_loss += micro_loss.item()

accum_grads = [p.grad.clone() for p in model.parameters()]

print('=== Gradient Accumulation ===')
print(f'Full batch loss: {full_loss:.6f}')
print(f'Accumulated loss: {accum_loss:.6f}')
print(f'Loss match: {abs(full_loss - accum_loss) < 1e-5}')

grad_diffs = []
for fg, ag in zip(full_grads, accum_grads):
    max_diff = (fg - ag).abs().max().item()
    grad_diffs.append(max_diff)
print(f'\nMax gradient difference: {max(grad_diffs):.8f}')
print(f'Gradients match (numerical precision): {max(grad_diffs) < 1e-5}')

print(f'\nEffective batch size = micro_batch({micro_bs}) x accum_steps({accum_steps}) = {micro_bs * accum_steps}')
print(f'\nKey: Gradient accumulation is mathematically equivalent to larger batch,')
print(f'but requires accum_steps forward passes per update step.')

## 3. 混合精度训练

**目的**：加速训练并减少显存占用

**基本原理**：使用低精度（FP16/BF16）进行前向和反向传播计算，同时维护一份FP32的主权重用于参数更新，在保持训练精度的同时大幅提升训练速度。

**混合精度训练流程**：
1. 维护FP32主权重（master weights）
2. 每步将权重转换为FP16/BF16用于前向传播
3. 前向和反向传播使用低精度计算
4. 将FP16梯度转回FP32更新主权重
5. 使用损失缩放（loss scaling）防止梯度下溢

**FP16 vs BF16**：
- FP16：范围小（6e-5~65504），需要损失缩放
- BF16：范围与FP32相同（1e-38~3e38），不需要损失缩放，现代GPU首选

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

W_fp32 = torch.randn(256, 256)
W_fp16 = W_fp32.half()
W_bf16 = W_fp32.bfloat16()

x = torch.randn(32, 256)

out_fp32 = x @ W_fp32.T
out_fp16 = x.half() @ W_fp16.T
out_bf16 = x.bfloat16() @ W_bf16.T

print('=== Mixed Precision Training ===')
print(f'\nMemory per element:')
print(f'  FP32: 4 bytes')
print(f'  FP16: 2 bytes (2x savings)')
print(f'  BF16: 2 bytes (2x savings)')

print(f'\nComputation speedup on modern GPUs:')
print(f'  FP16/BF16 matmul: ~2-4x faster than FP32')

print(f'\nPrecision comparison (vs FP32):')
fp16_error = (out_fp32 - out_fp16.float()).abs().mean().item()
bf16_error = (out_fp32 - out_bf16.float()).abs().mean().item()
print(f'  FP16 mean error: {fp16_error:.6f}')
print(f'  BF16 mean error: {bf16_error:.6f}')

print(f'\nValue range comparison:')
print(f'  FP16: max={torch.finfo(torch.float16).max:.0f} (~65504)')
print(f'  BF16: max={torch.finfo(torch.bfloat16).max:.0f} (same as FP32)')
print(f'  FP32: max={torch.finfo(torch.float32).max:.0f}')

print(f'\nKey: BF16 has the same range as FP32, so no loss scaling needed.')
print(f'FP16 has higher precision but smaller range, requiring loss scaling.')
print(f'Modern LLMs (LLaMA, GPT-4) use BF16 for training stability.')

## 4. FP8训练

**目的**：进一步提升训练效率

**基本原理**：使用8位浮点数进行矩阵乘法计算，相比BF16再提升约2倍计算速度，需要配合延迟缩放（delay scaling）等技术保持数值稳定性。

**FP8格式**：
- **E4M3**（4位指数+3位尾数）：用于前向传播，精度更高
- **E5M2**（5位指数+2位尾数）：用于反向传播，范围更大

**延迟缩放**：使用上一步的最大值来缩放当前步的FP8计算，避免实时计算缩放因子的开销

**硬件要求**：NVIDIA H100+或AMD MI300+，支持FP8 Tensor Core

In [ ]:
import torch
import struct

torch.manual_seed(42)

def simulate_fp8_e4m3(tensor, max_val=None):
    if max_val is None:
        max_val = tensor.abs().max()
    scale = 448.0 / max_val
    scaled = tensor * scale
    quantized = torch.clamp(torch.round(scaled), -448, 448)
    return quantized / scale, scale

def simulate_fp8_e5m2(tensor, max_val=None):
    if max_val is None:
        max_val = tensor.abs().max()
    scale = 57344.0 / max_val
    scaled = tensor * scale
    quantized = torch.clamp(torch.round(scaled), -57344, 57344)
    return quantized / scale, scale

W = torch.randn(256, 256)
x = torch.randn(32, 256)

out_fp32 = x @ W.T
W_e4m3, scale_w = simulate_fp8_e4m3(W)
x_e4m3, scale_x = simulate_fp8_e4m3(x)
out_fp8_fwd = x_e4m3 @ W_e4m3.T

W_e5m2, _ = simulate_fp8_e5m2(W)
out_fp8_bwd = x_e4m3 @ W_e5m2.T

print('=== FP8 Training ===')
print(f'\nFP8 Formats:')
print(f'  E4M3 (forward): 4-bit exponent + 3-bit mantissa')
print(f'    Max representable: ±448')
print(f'    Higher precision, smaller range')
print(f'  E5M2 (backward): 5-bit exponent + 2-bit mantissa')
print(f'    Max representable: ±57344')
print(f'    Larger range, lower precision')

fwd_error = (out_fp32 - out_fp8_fwd).abs().mean().item()
bwd_error = (out_fp32 - out_fp8_bwd).abs().mean().item()
fp32_norm = out_fp32.abs().mean().item()

print(f'\nQuantization error (vs FP32):')
print(f'  E4M3 forward: {fwd_error:.4f} (relative: {fwd_error/fp32_norm:.2%})')
print(f'  E5M2 backward: {bwd_error:.4f} (relative: {bwd_error/fp32_norm:.2%})')

print(f'\nMemory savings: FP8 uses 1 byte vs FP32 4 bytes = 4x reduction')
print(f'Compute speedup: ~2x vs BF16 on H100+ GPUs')
print(f'\nKey: FP8 is the frontier of training efficiency.')
print(f'Delay scaling uses previous step max values to avoid overhead.')